<a href="https://colab.research.google.com/github/natchanant-arch/Project_Savings_Cooperative/blob/%E0%B8%93%E0%B8%B1%E0%B8%90/Cooperative_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 ธุรกิจสหกรณ์ออมทรัพย์ (Savings Cooperative)

ระบบจำลองบัญชีสหกรณ์ออมทรัพย์ ครอบคลุมการ **ฝาก – ถอน – โอน** โดยสมาชิกทำธุรกรรมได้ทีละรายการ พร้อมตรวจสอบยอดเงินคงเหลือให้เพียงพอก่อนทำรายการทุกครั้ง และคำนวณดอกเบี้ยจากยอดเงินคงเหลือในบัญชีเมื่อสิ้นปี

---

## 📑 สารบัญ

| ส่วน | หัวข้อ |
|:---:|---|
| 0 | Import เพื่อเรียกใช้งานชุดคำสั่ง / ฟังก์ชันสำเร็จรูป |
| 1 | เตรียม Class และฟังก์ชัน |
| 2 | ทดสอบฟังก์ชันทีละตัว ก่อนประกอบเป็นกระบวนการ |
| 3 | ฟังก์ชันอธิบายขั้นตอนคำนวณรายการ |
| 4 | จำลอง "ลูกค้า 1 คนเดินเข้าธนาคาร" แบบ step-by-step |
| 5 | จำลองลูกค้าหลายคนเดินเข้าธนาคารต่อเนื่องกัน |
| 6 | สรุปผล — ยืนยันว่าฟังก์ชันคืนค่าถูกต้องและใช้ต่อได้จริง |
| 7 | ตารางลูกค้า |
| 8 | ตารางธุรกรรม |
| 9 | สรุปผลรวม 300 รายการ |

---

## — Import เพื่อ เรียกใช้งานชุดคำสั่ง หรือฟังก์ชันสำเร็จรูป —

In [ ]:
import random
import time
from datetime import datetime, timedelta
!pip install Faker
from faker import Faker
fake = Faker("th_TH")
random.seed(1)
import pandas as pd
import matplotlib, os, shutil
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

In [ ]:
# 1. ติดตั้งฟอนต์ภาษาไทย
!apt-get -y install fonts-thai-tlwg

# 2. ล้าง cache ของ matplotlib เพื่ออัปเดตฟอนต์ใหม่
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ใหม่เข้ากับ FontManager
font_path = '/usr/share/fonts/truetype/tlwg/Loma.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

# 4. ตั้งค่าฟอนต์หลัก
plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

print("ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!")

---

## ส่วนที่ 1 — เตรียม Class และฟังก์ชัน

มีทั้งหมด 3 คลาส คือ

1. **Class Member** (สมาชิก) — รหัสสมาชิก, ชื่อสมาชิก, เลขบัตรประจำตัวประชาชน, เบอร์โทรศัพท์
2. **Class Account** (บัญชีออมทรัพย์) — เลขบัญชี, ยอดเงินคงเหลือ, เจ้าของบัญชี, ดอกเบี้ยต่อปี
3. **Class Transaction** (ธุรกรรม) — หมายเลขธุรกรรม, บัญชี, ประเภทธุรกรรม, จำนวนเงิน, บัญชีปลายทาง

In [ ]:
class Member:
    """ข้อมูลสมาชิกธนาคารออมทรัพย์"""

    def __init__(
        self,
        member_id,
        customer_name,
        citizen_id=None,
        phone_number=None,
    ):
        self.member_id = member_id
        self.customer_name = customer_name
        self.citizen_id = citizen_id
        self.phone_number = phone_number

    # แสดงข้อมูลสมาชิก
    def get_info(self):
        return (
            f"ลูกค้า ID: {self.member_id} | ชื่อ: {self.customer_name} | "
            f"เลขบัตรประชาชน: {self.citizen_id} | เบอร์โทร: {self.phone_number}"
        )

    # อัปเดตข้อมูลส่วนตัว
    def update_phone(self, new_phone):
        self.phone_number = new_phone

In [ ]:
class Account:
    """บัญชีออมทรัพย์"""

    def __init__(
        self,
        account_number,
        balance,
        owner,
        interest_rate=0.015,
    ):
        self.account_number = account_number
        self.balance = float(balance)
        self.owner = owner
        self.interest_rate = interest_rate

    def deposit(self, amount):
        """ฝากเงิน: balance = balance + amount"""
        self.balance += amount
        return "ฝากเงินสำเร็จ"

    def withdraw(self, amount):
        """ถอนเงิน: ตรวจสอบ balance >= amount"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอ (มีอยู่ {self.balance:,.2f} บาท)"

        self.balance -= amount
        return "ถอนเงินสำเร็จ"

    def transfer(self, target_account, amount):
        """โอนเงิน: ตัดบัญชีต้นทาง และบวกเข้าบัญชีปลายทาง"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอโอน (มีอยู่ {self.balance:,.2f} บาท)"

        self.balance -= amount
        target_account.balance += amount
        return "โอนเงินสำเร็จ"

    def apply_interest(self):
        """คำนวณดอกเบี้ยและบวกเข้ายอดคงเหลือ"""
        interest = self.balance * self.interest_rate
        self.balance += interest
        return interest